In [ ]:
try:
  import google.colab
  IN_COLAB = True
except ImportError:
  IN_COLAB = False

if IN_COLAB:
  # Install dependencies
  ! pip install --upgrade pip
  ! pip install czitools

## Reading Metadata

For most workflows, create one `CziMetadata` object and use its grouped
properties. The metadata object returned by a pixel reader is the same type, so
there is no need to read it twice:

In [ ]:
from czitools.metadata_tools import CziMetadata

# enter the correct filepath here - it can be also a link to a czi file on the web, e.g. from github
#filepath = r"F:\Pixi_Projects\zen_czi\czitools\data\WP96_4Pos_B4-10_DAPI.czi"
filepath = r"/datadisk1/Github/czitools/data/WP96_4Pos_B4-10_DAPI.czi"

mdata = CziMetadata(filepath)

In [ ]:
image_dimensions = mdata.image
scale = mdata.scale
channels = mdata.channelinfo

print(f"SizeS: {image_dimensions.SizeS}")
print(f"SizeT: {image_dimensions.SizeT}")
print(f"SizeZ: {image_dimensions.SizeZ}")
print(f"SizeC: {image_dimensions.SizeC}")
print(f"SizeY: {image_dimensions.SizeY}")
print(f"SizeX: {image_dimensions.SizeX}")
print(f"Physical Pixel Size XYZ: {scale.X} x {scale.Y} x {scale.Z}")
print(f"Pixel types: {mdata.pixeltypes}")

The `*_required` properties return a non-optional value or raise a clear error.
Frequently used groups are:

| Property | Contents |
| --- | --- |
| `image` | Dimension sizes |
| `bbox` | Total and per-scene bounding boxes |
| `channelinfo` | Channel acquisition and display information |
| `scale` | Physical scaling |
| `objective`, `detector`, `microscope` | Instrument information |
| `sample` | Per-scene sample and well information |
| `attachments` | CZI attachment availability |
| `hcs`, `hcs_status` | Validated HCS hierarchy and detection result |

Metadata can also be read selectively using the individual classes:

In [ ]:
from czitools.metadata_tools.dimension import CziDimensions
from czitools.metadata_tools.channel import CziChannelInfo
from czitools.metadata_tools.scaling import CziScaling
from czitools.metadata_tools.sample import CziSampleInfo
from czitools.metadata_tools.objective import CziObjectives
from czitools.metadata_tools.microscope import CziMicroscope
from czitools.metadata_tools.add_metadata import CziAddMetaData
from czitools.metadata_tools.detector import CziDetector

# Image Dimensions
czi_dimensions = CziDimensions(filepath)

# Channel information
czi_channels = CziChannelInfo(filepath)

# Scaling (values in microns)
czi_scale = CziScaling(filepath)

# Objectives, detectors, microscope
czi_objectives = CziObjectives(filepath)
czi_detectors = CziDetector(filepath)
czi_microscope = CziMicroscope(filepath)

# Sample carrier info
czi_sample = CziSampleInfo(filepath)

# Additional metadata
czi_addmd = CziAddMetaData(filepath)

## Reading Pixel Data

Choose the reader according to the regularity of the data and whether loading
must be genuinely lazy:

| Requirement | Recommended function | Result |
| --- | --- | --- |
| Equal-sized scenes; eager read | `read_6darray` | One `STCZYX(A)` array |
| Regular array wrapped as Dask | `read_6darray(use_dask=True)` | Dask-backed, but eagerly read |
| True on-demand reads | `read_stacks(use_dask=True)` | Per-scene arrays by default |
| True lazy reads with equal scenes stacked | `read_stacks_stacked` | One array with `S` |
| Scenes that may differ in shape | `read_stacks_list` | Stable list of scene arrays |
| A known HCS well or field | `read_well` / `read_field` | HCS-aware field arrays |

### `read_6darray` — Full 6D Stack

Returns the image as a single array with dimension order **STCZYX(A)**.
It requires equal-sized scenes and consistent pixel types:

In [ ]:
from czitools.read_tools import read_6darray

array6d, mdata = read_6darray(filepath, use_dask=True, use_xarray=True, zoom=1.0)

# show the info about the array
array6d

In [ ]:
# Read zero-based inclusive S/T/C/Z ranges
subset, mdata = read_6darray(
    filepath,
    use_dask=True,
    use_xarray=True,
    planes={"S": (0, 3), "T": (0, 0), "C": (0, 0), "Z": (0, 0)},
    adapt_metadata=True,
    zoom=0.5
)

# show the info about the array
subset

### `read_stacks` — Scene-Wise Reading

`read_stacks` supports all CZI dimensions and optionally stacks compatible
scenes. With `use_dask=True`, pixel planes are read only when indexed or
computed:

In [ ]:
from czitools.read_tools import read_stacks

result, dims, num_stacks, mdata = read_stacks(
    filepath,
    use_dask=True,
    use_xarray=True,
    stack_scenes=True,   # attempt to stack all scenes into one array
)

result

Return behaviour:

| `stack_scenes` | Scenes compatible? | Return type                         |
| -------------- | ------------------ | ----------------------------------- |
| `False`        | —                  | `list` (one array per scene)        |
| `True`         | Yes                | Single stacked array (with `S` dim) |
| `True`         | No                 | `list` (with warning)               |

For strict return contracts:

In [ ]:
from czitools.read_tools import read_stacks_list, read_stacks_stacked

# Always returns a list
result_list, dims, n, mdata = read_stacks_list(
    filepath,
    use_dask=True,
)

# Raises ValueError if scenes cannot be stacked
stacked, dims, n, mdata = read_stacks_stacked(
    filepath,
    use_dask=True,
)

`read_stacks_list` is the safest interface for files whose scenes may have
different shapes. Call `.compute()` on a Dask-backed selection when its pixels
are needed.

For example, this reads only one selected plane:

In [ ]:
scenes, dims, scene_count, mdata = read_stacks_list(
    filepath,
    use_dask=True,
    use_xarray=True,
    planes={"T": (0, 0), "C": (0, 0)},
)

first_plane = scenes[0].isel(T=0, C=0, Z=0).compute()

# show 1st scene
scenes[0]


## Reading Well-Plate Metadata

`CziMetadata` provides both an HCS detection result and, when detection is
successful, an immutable `Plate -> Well -> Field` hierarchy:

In [ ]:
from czitools.metadata_tools import CziMetadata
from czitools.metadata_tools.hcs import (
    resolve_field,
    resolve_well,
    well_relative_field_positions,
    well_absolute_field_positions
)
from czitools.read_tools import read_field, read_well
import matplotlib.pyplot as plt
import numpy as np

mdata = CziMetadata(filepath)
sample = mdata.sample
plate = mdata.hcs
selected_well = "B4"
tolerance = 1.0

print("\nPlate:")
print(f"  ID: {plate.id}")
print(f"  Name: {plate.name}")
print(f"  File: {filepath}")
print(f"  HCS detected: {mdata.hcs_status.detected}")
print(f"  HCS Reason: {mdata.hcs_status.reason}")
print(f"  Scene count: {sample.scene_count}")
print(f"  Unique well count: {sample.well_unique_number}")
print(f"  Multiple fields per well: {sample.multipos_per_well}")
print(f"  Wells present in this CZI: {len(plate.wells)}")
print(f"  Fields present in this CZI: {sum(len(well.fields) for well in plate.wells)}")


`CziMetadata` provides both an HCS detection result and, when detection is
successful, an immutable `Plate -> Well -> Field` hierarchy:

In [ ]:
if mdata.hcs is not None:
    plate = mdata.hcs
    print(f"PlateID: {plate.id}, Name: {plate.name}, Schema Version: {plate.schema_version}")
    print(f"Rows (declared): {plate.declared_rows}, Columns (declared): {plate.declared_columns}")
    print(f"Rows (observed): {plate.observed_row_indices}")       # normalized, zero-based
    print(f"Columns (observed): {plate.observed_column_indices}")    # normalized, zero-based

    for well in plate.wells:
        print(
            f"Canonical name: {well.canonical_name}",
            f"Canonical path: {well.canonical_path}",
            f"CZI=({well.source_row_index}, {well.source_column_index})",
            f"Row: {well.row_index}, Column: {well.column_index}",
            f"Fields: {len(well.fields)}",
        )

    # Capitalization, zero padding, and path notation are normalized.
    well = plate.get_well("b04")
    for field in well.fields:
        print(
            f"Field: {field.field_index}",       # zero-based within the well
            f"Scene: {field.scene_index}",       # global CZI scene
            f"Region ID: {field.region_id}",
            f"Scene Center X: {field.scene_center_x}",
            f"Scene Center Y: {field.scene_center_y}",
        )

In [ ]:
if sample.scene_count:
    for s in range(sample.scene_count):
        well_name = sample.well_array_names[s] or "<missing>"
        region_id = sample.well_region_ids[s]
        center_x = sample.field_centerX[s]
        center_y = sample.field_centerY[s]

    # plot all cneter XY positions as a simple grid
    plt.figure(figsize=(6, 6))
    plt.scatter(sample.field_centerX, sample.field_centerY, marker="o", color="blue", label="Field Centers")
    plt.title("Field Center Positions")
    plt.xlabel("Center X (micrometers)")
    plt.ylabel("Center Y (micrometers)")
    plt.grid()
    plt.axis('equal')
    plt.legend()
    plt.show()  

The model retains the original CZI well indices as well as normalized
zero-based indices. Field indices are local to a well; scene indices are global
to the CZI.

Resolver functions map selectors to the model without reading pixels:

In [ ]:
from czitools.metadata_tools.hcs import resolve_field, resolve_well

well = resolve_well(plate, "B/04")
field = resolve_field(plate, "B/04", 0)

print(f"Field Region ID: {field.region_id}")

# A source RegionId string can select the same field.
if field.region_id is not None:
    same_field = resolve_field(plate, "B04", field.region_id)

print("------------------------------------------")

# show some parameters of the field
print(f"Field Region ID: {same_field.id}")
print(f"Position Name: {same_field.position_name}")
print(f"Field Index: {same_field.field_index}")


The compatibility-oriented `mdata.sample` object exposes per-scene
collections.

Prefer `sample.field_centerX` and `sample.field_centerY`: they
preserve valid `0.0` coordinates and use `None` for missing positions.

The
deprecated `scene_stageX` and `scene_stageY` properties convert missing values
to `0.0`.

HCS detection is an additional interpretation. If `mdata.hcs` is `None`,
general metadata and ordinary CZI pixel reading remain available.

### Optional stage-position enrichment

Scene-center positions come from scene XML. Subblock stage/focus coordinates
can be added explicitly from the planetable for a local CZI:

In [ ]:
enriched_plate = mdata.enrich_hcs_positions(position_tolerance=1.0)

if enriched_plate is not None:
    well = enriched_plate.get_well("B04")
    for field in well.fields:
        print(
            f"Field StageX: {field.stage_x}",
            f"Field StageY: {field.stage_y}",
            f"Acquisition Z: {field.acquisition_z}",
            f"Position Conflict: {field.position_conflict}",
        )

> Important: Enrichment returns a new immutable plate and updates `mdata.hcs`.
> It keeps scene-center and subblock-stage coordinates separate and is unavailable for URL
> sources.

Position helpers expose well-relative scene-center offsets and absolute
coordinates. They return `None` if any required coordinate is missing:

In [ ]:
from czitools.metadata_tools.hcs import (
    well_absolute_field_positions,
    well_relative_field_positions,
)

if enriched_plate is None:
    raise ValueError("Stage positions are unavailable.")

well = enriched_plate.get_well("B04")
relative = well_relative_field_positions(well)
scene_centers = well_absolute_field_positions(well, source="scene_center")
stage_positions = well_absolute_field_positions(well, source="stage")

# show all well attributes
for parameter in well.__dict__:
    print(f"{parameter}: {getattr(well, parameter)}")

print("---------   Show individual field parameter   ------------")

for fp in well.fields[0].__dict__:
    print(f"{fp}: {getattr(well.fields[0], fp)}")


## Reading by Well / Field (HCS Plates)

For high-content-screening plates, wells and fields can be read directly by name
without tracking scene indices. These reads use the canonical HCS model
(`CziMetadata.hcs`) and reuse the single-scene read path.

In [ ]:
from czitools.read_tools import read_field, read_well

# Read a single field of a well (well names accept "B4", "b04" or "B/4").
# `field` is the well-local 0-based index, or a source-scoped RegionId string.
array, mdata = read_field(filepath, well="B4", field=0)

# Read all fields of a well as a list of per-field arrays (shapes may differ).
arrays, mdata = read_well(filepath, well="B4")

# Stack the fields along the S axis (requires identical field shapes).
stacked, mdata = read_well(filepath, well="B4", stack=True)

If the CZI has no usable HCS plate metadata, both functions raise a `ValueError`
explaining why (from `CziMetadata.hcs_status.reason`).

These functions reuse `read_6darray`. Their `use_dask=True` results are
Dask-backed, but are not true on-demand reads from the CZI.

In [ ]:
# get_well normalizes capitalization and zero padding, so B04 resolves to B4.
print(f"\nFields in requested well {selected_well!r} -> {well.canonical_name}:")
for field in well.fields:
    print(
        f"  local={field.field_index} scene={field.scene_index} id={field.id} "
        f"region={field.region_id} center=({field.scene_center_x}, {field.scene_center_y}) "
        f"{field.position_unit}"
        )

In [ ]:
# Pure resolution (no pixel reading). resolve_well normalizes the name.
well = resolve_well(plate, selected_well)

print(
    f"Resolved well {selected_well!r} -> {well.canonical_name} with {len(well.fields)} field(s)."
)

# Resolve the first field in the well (field_index=0) and print its details.
first_field = resolve_field(plate, selected_well, 0)

print(
    f"field 0 -> scene={first_field.scene_index} id={first_field.id} region={first_field.region_id}"
)

In [ ]:
# Read a single field (field 0) -> one scene, returned as an xarray.DataArray.
print(f"\nread_field({well.canonical_name!r}, 0):")

array, _ = read_field(filepath, selected_well, 0, use_xarray=True)

if array is not None:
    print(f"shape={array.shape} dims={getattr(array, 'dims', None)}")

    plt.figure()
    plt.imshow(array[0, 0, 0, 0, ...], cmap="gray")
    plt.title(f"Well {well.canonical_name} field 0")
    plt.show()

In [ ]:
# Read all fields of the well as a list (fields may differ in shape).
print(f"\nread_well({well.canonical_name!r}) -> list of per-field arrays:")

arrays, _ = read_well(filepath, selected_well)

print(
    f"Type of arrays: {type(arrays)}, Length: {len(arrays) if arrays is not None else 'N/A'}"
)

if isinstance(arrays, list):
    for index, field_array in enumerate(arrays):
        print(f"  field {index}: shape={field_array.shape}")

if arrays and len(arrays) > 0:
    n_fields = len(arrays)
    n_cols = int(np.ceil(np.sqrt(n_fields)))
    n_rows = int(np.ceil(n_fields / n_cols))

    fig, axes = plt.subplots(n_rows, n_cols, figsize=(4 * n_cols, 4 * n_rows))
    axes = np.array(axes).flatten()

    for idx, field_array in enumerate(arrays):
        axes[idx].imshow(field_array[0, 0, 0, 0, ...], cmap="gray")
        axes[idx].set_title(f"Well {well.canonical_name} field {idx}")
        axes[idx].axis("off")

    for idx in range(n_fields, len(axes)):
        axes[idx].set_visible(False)

    plt.tight_layout()
    plt.show()

## Array Dimension Order

`read_6darray`, `read_field`, and `read_well` use **STCZYX(A)**:

| Dim | Meaning                         |
| --- | ------------------------------- |
| S   | Scene                           |
| T   | Time                            |
| C   | Channel                         |
| Z   | Z-slice                         |
| Y   | Y (height)                      |
| X   | X (width)                       |
| A   | RGB sample/component (optional) |

`read_stacks` tracks `S` separately unless scenes are stacked. Optional CZI
dimensions `V`, `R`, `I`, `H`, and `M` precede the always-present core
dimensions `T`, `C`, and `Z`; spatial dimensions and optional `A` follow.

## Exporting to OME-Zarr

!!! note "Requires the `omezarr` extra"
    OME-Zarr export lives in `czitools.export_tools` and needs the optional
    dependencies: `pip install "czitools[omezarr]"` (or `"czitools[omezarr-gui]"`
    for the GUI). See the [Installation docs](install.md).

### HCS plate export

The converter writes the logical hierarchy plate → row/column well → field
image → multiscale level. It resolves the layout from `CziMetadata.hcs` and
uses complete, unambiguous sample metadata only as a fallback. Fields are
written individually, so fields with different shapes are supported.

In [ ]:
from pathlib import Path

from czitools.export_tools import (
    convert_czi2hcs_ngff,  # ngff-zarr backend, OME-NGFF v0.5
    convert_czi2hcs_omezarr,  # ome-zarr-py backend, Zarr v3 by default
    validate_ome_zarr,
)

# Write an HCS plate (rows/wells/fields) with the ngff-zarr backend.
out = convert_czi2hcs_ngff(
    filepath,
    output_dir=Path("exports"),
    overwrite=True,
    pad_columns=True,
)

# Validate the result against the OME-NGFF v0.5 schema.
assert validate_ome_zarr(out)